In [2]:
# !pip install bayesian-optimization
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
from bayes_opt import BayesianOptimization
import matplotlib.pyplot as plt

# ==============================
# 1. 黑盒 CNN 模型
# ==============================
class BlackBoxModel(nn.Module):
    def __init__(self):
        super(BlackBoxModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)  # 16x16
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)  # 8x8
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.softmax(x, dim=1)  # 只返回 Softmax 概率

# 检查 GPU 是否可用，训练阶段使用 GPU，加速训练
train_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
attack_device = torch.device("cpu")  # 贝叶斯优化仅支持 CPU

# 初始化黑盒模型
model = BlackBoxModel().to(train_device)

# ==============================
# 2. 训练黑盒模型（使用 GPU）
# ==============================
transform = transforms.Compose([transforms.ToTensor()])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"开始训练黑盒模型（使用 {train_device}）...")
for epoch in range(5):
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(train_device), labels.to(train_device)
        optimizer.zero_grad()
        outputs = model(inputs)  # 只返回 Softmax 概率
        loss = criterion(torch.log(outputs + 1e-9), labels)  # 计算交叉熵损失
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(trainloader):.4f}")

print("黑盒模型训练完成！")

# **训练完成后，将模型转换到 CPU 以支持贝叶斯优化**
model.to(attack_device)


开始训练黑盒模型（使用 cuda）...
Epoch 1, Loss: 1.6052
Epoch 2, Loss: 1.2717
Epoch 3, Loss: 1.1314
Epoch 4, Loss: 1.0346
Epoch 5, Loss: 0.9629
黑盒模型训练完成！


BlackBoxModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

In [ ]:
from joblib import Parallel, delayed
from sklearn.decomposition import PCA

# ==============================
# 1. 黑盒模型 API
# ==============================
def blackbox_model(x):
    """ 通过黑盒 API 获取 Softmax 概率 """
    with torch.no_grad():
        x = x.to(attack_device)
        probs = model(x)
    return probs

def estimate_logits(softmax_probs):
    """ 估算 Logits（从 Softmax 概率反推）"""
    log_probs = torch.log(softmax_probs + 1e-9)  # 防止 log(0)
    return log_probs - log_probs.mean()  # 归一化，去除未知的 C

# ==============================
# 2. 维度降维（PCA）
# ==============================
def reduce_dimensionality(delta, pca):
    """ 使用 PCA 进行降维 """
    delta_flat = delta.reshape(1, -1)
    return pca.transform(delta_flat)

def restore_dimensionality(delta_reduced, pca):
    """ 恢复降维后的变量到原始维度 """
    delta_restored = pca.inverse_transform(delta_reduced)
    return delta_restored.reshape(1, 3, 32, 32)

# ==============================
# 3. 贝叶斯优化攻击目标函数
# ==============================
def adversarial_success(pca, x, y, **delta_dict):
    """
    计算目标函数：
    - 目标是找到最小 delta，使得 model(x + delta) ≠ y
    - 这里使用 Logit 差值作为优化目标
    """
    # ✅ **转换 `delta` 为 `float32`**
    # delta_values = [delta_dict[f'delta_{i}'] for i in range(3 * 32 * 32)]
    # delta = torch.tensor(delta_values, dtype=torch.float32).view(1, 3, 32, 32).to(x.device)

    delta_reduced = np.array([delta_dict[f'delta_{i}'] for i in range(50)])  # 降维到 50 维
    delta = torch.tensor(restore_dimensionality(delta_reduced, pca), dtype=torch.float32).to(x.device)
    

    x_adv = x + delta  # 生成对抗样本
    x_adv = x_adv.to(torch.float32)  # ✅ **确保 `x_adv` 是 `float32`**
    x_adv = torch.clamp(x_adv, 0, 1)  # 限制像素范围在 [0, 1] 之间

    softmax_probs = blackbox_model(x_adv)  # 获取 API 返回的 Softmax 概率
    logits = estimate_logits(softmax_probs)  # 估算 Logits

    correct_logit = logits[0, y.item()]  # 真实类别的 Logit
    max_wrong_logit = logits[0].max()  # 最高错误类别的 Logit

    # if max_wrong_logit > correct_logit:  # 误分类成功
    #     return -1  # 目标最优值
    # else:
    #     return correct_logit - max_wrong_logit  # 继续优化
    confidence = 0.2  # 允许小误差，提高攻击成功率
    if max_wrong_logit > correct_logit - confidence:
        return -1  # 误分类成功
    else:
        return -1 * (correct_logit - max_wrong_logit)  # 让 loss 越小越好

# ==============================
# 4. 贝叶斯优化攻击函数
# ==============================
def bayesian_attack(model, x, y, max_queries=50):
    """
    使用贝叶斯优化搜索最小扰动 delta
    """

    delta_range = 0.9

    x = x.clone().detach().to("cpu").to(torch.float32)  # ✅ **确保 `x` 是 `float32`**
    y = y.clone().detach().to("cpu")

    # pbounds = {f'delta_{i}': (-0.3, 0.3) for i in range(3 * 32 * 32)}
    # global pca  # 共享 PCA 变量
    num_samples = 100  # 生成 100 个扰动样本
    delta_samples = np.random.uniform(-delta_range, delta_range, (num_samples, 3 * 32 * 32))

    pca = PCA(n_components=min(50, num_samples))  # 动态选择 n_components
    pca.fit(delta_samples)  # 确保 PCA 训练成功

    pbounds = {f'delta_{i}': (-delta_range, delta_range) for i in range(50)}  # 降维到 50 维
    
    optimizer = BayesianOptimization(
        f=lambda **delta_dict: adversarial_success(pca, x, y, **delta_dict),
        pbounds=pbounds,
        random_state=42,
        verbose=0,  # 0 = 不输出日志, 1 = 仅输出结果, 2 = 输出所有信息
    )

    # optimizer.maximize(init_points=10, n_iter=max_queries)
    # 🔥 提前终止策略（Early Stopping）
    best_so_far = None
    patience = 10  # 如果 10 轮没有改善，提前停止
    no_improvement = 0

    for _ in range(max_queries):
        optimizer.maximize(init_points=10, n_iter=1)
        new_best = optimizer.max["target"]

        if best_so_far is None or new_best < best_so_far:
            best_so_far = new_best
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            print("🚀 触发提前终止，优化停止！")
            break

    # best_delta = torch.tensor([optimizer.max['params'][f'delta_{i}'] for i in range(3 * 32 * 32)], dtype=torch.float32).view(1, 3, 32, 32)
    # x_adv = (x + best_delta.to(x.device)).to(torch.float32)  # ✅ **确保 `x_adv` 是 `float32`**

    best_delta_reduced = np.array([optimizer.max['params'][f'delta_{i}'] for i in range(50)])
    best_delta = torch.tensor(restore_dimensionality(best_delta_reduced, pca), dtype=torch.float32)
    x_adv = (x + best_delta.to(x.device)).to(torch.float32)
    x_adv = torch.clamp(x_adv, 0, 1)  # 限制像素范围

    return x_adv



# ==============================
# 5. 运行测试（使用 CPU）
# ==============================
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)

# x, y = next(iter(testloader))
# x, y = x.to(attack_device), y.to(attack_device)

# print(f"原始类别: {y.item()}")
# x_adv = bayesian_attack(model, x, y, max_queries=500)

# ==============================
# 6. 评估 & 可视化攻击结果
# ==============================
def evaluate_attack(model, x, x_adv, y):
    """
    评估攻击是否成功
    """
    softmax_probs_x = blackbox_model(x)
    softmax_probs_x_adv = blackbox_model(x_adv)

    pred_x = softmax_probs_x.argmax(dim=1).item()
    pred_x_adv = softmax_probs_x_adv.argmax(dim=1).item()

    print(f"原始预测类别: {pred_x}, 对抗样本预测类别: {pred_x_adv}")
    if pred_x != pred_x_adv:
        print("🎯 攻击成功！")
        return True
    else:
        print("❌ 攻击失败！")
        return False

def visualize_attack(x, x_adv):
    """
    绘制原始样本和对抗样本
    """
    x_np = x.squeeze(0).permute(1, 2, 0).cpu().numpy()
    x_adv_np = x_adv.squeeze(0).permute(1, 2, 0).cpu().numpy()

    fig, axs = plt.subplots(1, 3, figsize=(10, 5))
    axs[0].imshow(np.clip(x_np, 0, 1))
    axs[0].set_title("Original Image")
    
    axs[1].imshow(np.clip(x_adv_np, 0, 1))
    axs[1].set_title("Adversarial Image")

    axs[2].imshow(np.abs(x_np - x_adv_np))
    axs[2].set_title("Perturbation (Diff)")
    
    plt.show()

# 评估攻击效果
# evaluate_attack(model, x, x_adv, y)
# visualize_attack(x, x_adv)

# 做10次攻击尝试，评估攻击效果并统计成功率
success_count = 0
for _ in range(10):
    x, y = next(iter(testloader))
    x, y = x.to(attack_device), y.to(attack_device)
    x_adv = bayesian_attack(model, x, y, max_queries=500)
    if evaluate_attack(model, x, x_adv, y):
        success_count += 1
    visualize_attack(x, x_adv)

print(f"攻击成功率: {success_count / 10 * 100:.2f}%")
# ==============================



🚀 触发提前终止，优化停止！
原始预测类别: 3, 对抗样本预测类别: 3
❌ 攻击失败！


RuntimeError: a Tensor with 1024 elements cannot be converted to Scalar